In [1]:
# !pip install seaborn
# !pip install yellowbrick
# !pip install geopandas
# !pip install openpyxl

In [1]:
import pandas as pd
import geopandas as gpd

In [6]:
df = pd.read_csv('C:/Users/hyeon/OneDrive/Desktop/2026-1/공모전/데이터분석/데이터/반지하/서울시_주택인허가_동별개요.csv', encoding = 'cp949')
df.head()

C:\Users\hyeon\AppData\Local\Temp\ipykernel_32672\3457126806.py:1: DtypeWarning: Columns (11,37) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('C:/Users/hyeon/OneDrive/Desktop/2026-1/공모전/데이터분석/데이터/반지하/서울시_주택인허가_동별개요.csv', encoding = 'cp949')


,대지위치,시군구코드명,법정동코드명,대지구분코드명,주지번,부지번,동별개요일련번호,주택대장일련번호,건물명,특수지명,...,지상층수,높이,승용승강기인용수정보,비상용승강기인용수정보,층시작높이값,반자높이값,계단유효너비,복도너비,외벽두께,인접세대벽두께
0,서울특별시 성북구 하월곡동,서울특별시 성북구,하월곡동,NaN,0.0,0.0,1007404,100726,샹그레빌 아파트,NaN,...,1.0,3.45,0.0,0.0,0.0,2.5,0.0,0.0,25.0,0.0
1,서울특별시 성북구 하월곡동,서울특별시 성북구,하월곡동,NaN,0.0,0.0,1007403,100726,샹그레빌 아파트,NaN,...,1.0,2.40,0.0,0.0,2.4,2.14,0.0,0.0,25.0,0.0
2,서울특별시 성북구 하월곡동,서울특별시 성북구,하월곡동,NaN,0.0,0.0,1007402,100726,샹그레빌 아파트,NaN,...,1.0,2.40,0.0,0.0,2.4,2.14,0.0,0.0,25.0,0.0
3,서울특별시 성북구 하월곡동,서울특별시 성북구,하월곡동,NaN,0.0,0.0,1007401,100726,샹그레빌 아파트,NaN,...,0.0,11.30,0.0,0.0,0.0,2.9,0.0,0.0,30.0,0.0
4,서울특별시 성북구 하월곡동,서울특별시 성북구,하월곡동,NaN,0.0,0.0,1007400,100726,샹그레빌 아파트,NaN,...,0.0,6.00,0.0,0.0,0.0,5.2,0.0,0.0,0.0,0.0


In [7]:
print(df.shape)
print(df.columns)

(58372, 42)
Index(['대지위치', '시군구코드명', '법정동코드명', '대지구분코드명', '주지번', '부지번', '동별개요일련번호',
       '주택대장일련번호', '건물명', '특수지명', '블록번호', '로트번호', '주부속구분코드명', '동명', '주용도코드명',
       '국민임대세대수', '공공임대5년세대수', '공공임대10년세대수', '공공임대기타세대수', '공공임대총세대수',
       '공공분양세대수', '사원임대세대수', '근로복지세대수', '민간임대세대수', '민간분양세대수', '구조코드명', '지붕코드명',
       '건축면적', '연면적', '지하면적', '용적률산정연면적', '지하층수', '지상층수', '높이', '승용승강기인용수정보',
       '비상용승강기인용수정보', '층시작높이값', '반자높이값', '계단유효너비', '복도너비', '외벽두께', '인접세대벽두께'],
      dtype='object')


In [8]:
# 지하층수 중복 요소 수 확인
df['지하층수'].value_counts()

지하층수
 0.0     15575
 1.0     11053
 2.0      6136
 3.0      2755
 4.0      1187
 5.0       557
 6.0       203
 7.0       169
 8.0        88
 9.0        13
 15.0        3
 16.0        3
 13.0        2
 10.0        2
 20.0        2
 18.0        2
 14.0        1
 21.0        1
-1.0         1
 19.0        1
 23.0        1
 12.0        1
Name: count, dtype: int64

In [9]:
# 주거용 건물 항목 필터링 (지하층수 0을 지우지 않은 상태에서 수행)
types = ['공동주택', '아파트', '단독주택']
df = df[df['주용도코드명'].isin(types)]

In [10]:
df = df[['시군구코드명', '법정동코드명', '주지번', '부지번', '지하층수', '지하면적']]

In [11]:
df['시군구코드명'] = df['시군구코드명'].str.strip()
df['법정동코드명'] = df['법정동코드명'].str.strip()
df['법정동명'] = df['시군구코드명'] + " " + df['법정동코드명']

In [12]:
code = pd.read_excel('C:/Users/hyeon/OneDrive/Desktop/2026-1/공모전/데이터분석/데이터/행정구역/법정동코드 조회자료.xlsx',
                     dtype = {'법정동코드' : str})
code['법정동명'] = code['법정동명'].str.strip()

c:\Users\hyeon\anaconda3\envs\cv_final\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [13]:
final_df = pd.merge(df, code[['법정동명', '법정동코드']], on='법정동명', how='left')

In [14]:
# 1. 법정동별 전체 주거용 건물 수 집계
total_counts = final_df.groupby('법정동코드').size().reset_index(name='전체_건물수')

# 2. 법정동별 반지하 건물 수 집계 (지하층수가 1 이상인 건물만 필터링)
basement_df = final_df[(final_df['지하층수'] != 0.0) & (final_df['지하층수'] != -1.0)]
basement_counts = basement_df.groupby('법정동코드').size().reset_index(name='반지하_개수')

In [15]:
# 3. 전체 건물 수 데이터와 반지하 건물 수 데이터 병합
analysis_df = pd.merge(total_counts, basement_counts, on='법정동코드', how='left')

# 4. 반지하 건물이 없는 동은 NaN이 되므로 0으로 채우기
analysis_df['반지하_개수'] = analysis_df['반지하_개수'].fillna(0)

# 5. 비율(%) 또는 소수점 비율 계산
analysis_df['반지하_비율'] = analysis_df['반지하_개수'] / analysis_df['전체_건물수']

In [16]:
# 결과 확인
analysis_df.head()

,법정동코드,전체_건물수,반지하_개수,반지하_비율
0,1111010100,13,13.0,1.000000
1,1111011100,13,0.0,0.000000
2,1111011500,34,19.0,0.558824
3,1111015900,1,1.0,1.000000
4,1111016700,1,1.0,1.000000


In [18]:
gdf = gpd.read_file("C:/Users/hyeon/OneDrive/Desktop/2026-1/공모전/데이터분석/데이터/행정구역/행정구역_법정동_경계/admstr_zone_lgldong_bndry_24.shp", encoding='cp949')
gdf.head()

,EMD_CD,COL_ADM_SE,EMD_NM,SGG_OID,geometry
0,11110101,11110,청운동,1036,"POLYGON ((196593.774 554114.361, 196611.342 55..."
1,11110102,11110,신교동,307,"POLYGON ((196912.513 554003.597, 196920.988 55..."
2,11110103,11110,궁정동,1034,"POLYGON ((197377.934 553847.046, 197388.359 55..."
3,11110104,11110,효자동,309,"POLYGON ((197664.978 553742.317, 197663.825 55..."
4,11110105,11110,창성동,316,"POLYGON ((197662.098 553585.836, 197662.501 55..."


In [20]:
# 8자리 키 생성 및 병합
gdf['merge_key'] = gdf['EMD_CD'].astype(str).str[:8]
analysis_df['merge_key'] = analysis_df['법정동코드'].astype(str).str[:8]

merged_gdf = gdf.merge(analysis_df, on='merge_key', how='left')

In [21]:
# 결합 후 비율이 없는 지역은 0으로 채우기
merged_gdf['반지하_비율'] = merged_gdf['반지하_비율'].fillna(0)
merged_gdf['전체_건물수'] = merged_gdf['전체_건물수'].fillna(0)
merged_gdf['반지하_개수'] = merged_gdf['반지하_개수'].fillna(0)

In [22]:
# 시각화 실행 (column을 '반지하_비율'로 변경)
m = merged_gdf.explore(
    column='반지하_비율',        
    cmap='YlOrRd',                       
    tooltip=['EMD_NM', '전체_건물수', '반지하_개수', '반지하_비율'], # 마우스 올렸을 때 비율도 함께 표시
    style_kwds={'weight': 0.5, 'color': 'gray', 'fillOpacity': 0.8}
)

In [23]:
m.save("seoul_basement_ratio_map.html")